# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>

# <font color="#003660">Session 5: Retrieval-Augmented Generation</font>

# <font color="#003660">Answering Questions with Documents</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... can answer questions about a document by putting it into the prompt, <br>
        ... know what the model sees when you send a document with a prompt template, <br>
        ... can do the same with LangChain and <code>ChatOpenAI</code>, <br>
        ... know why this stops working for long documents, and why we need RAG.
    </font>
</div>
</p>

Inspired by these excellent sources:

* [HuggingFace (2024): NLP Course](https://huggingface.co/learn/nlp-course/)
* [Prompt Engineering Guide](https://www.promptingguide.ai/)

Documentation for this notebook:

* [LangChain Docs: ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai)
* [LangChain Docs: Build a RAG agent](https://docs.langchain.com/oss/python/langchain/rag)
* [LangChain API Reference](https://reference.langchain.com/python/)
* [AI-Gateway of Paderborn University](https://hilfe.uni-paderborn.de/AI-Gateway)

## Where We Are

* **4.03:** the OpenAI-compatible API and the AI-Gateway.
* **4.04:** structured prompts with `###` blocks and delimiters.
* **Now:** give the model a **document** and let it answer questions about it.

This is the first step towards **retrieval-augmented generation (RAG)**, the topic of this session.

## Setup

The same setup as in 4.03: your own computer, connected to the university network or the VPN, and your AI-Gateway key.

**Good to know:** the gateway allows about 10 requests per minute per model. If a cell pauses, the client is waiting for the next minute and then continues by itself.

In [ ]:
%pip install -q openai langchain langchain-openai python-dotenv

In [ ]:
import os
import time
import urllib.request

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads the .env file: searches this folder first, then the folders above (the repo folder)
API_KEY = os.getenv("AI_GATEWAY_KEY")
AI_GATEWAY_BASE = os.getenv("AI_GATEWAY_BASE")  # address of the university's AI-Gateway
if not API_KEY or not AI_GATEWAY_BASE:
    raise ValueError("AI_GATEWAY_KEY or AI_GATEWAY_BASE missing. Copy .env.example to .env in the repo folder and paste your key there.")
MODEL = "gwdg.qwen3.8-27b-no-think"

client = OpenAI(base_url=AI_GATEWAY_BASE, api_key=API_KEY)

The key comes from the `.env` file in the repo folder (set up in 4.03). If the cell says that no key was found, copy `.env.example` to `.env` and paste your key behind `AI_GATEWAY_KEY=`.

## Part 1: A Question the Model Cannot Know

**In short: a model only knows what was in its training data, and what is in the prompt.**

Below is the travel policy of a company. The company and the policy are made up, so the model has never seen them.

In [ ]:
POLICY = """LUMEN ANALYTICS GMBH - TRAVEL POLICY (valid from 1 January 2026)
This is a fictional document for teaching purposes.

1. Approval: Every business trip must be approved by your team lead at least 5 working days in advance. Trips abroad also need the approval of the head of department.
2. Train or plane: Within Germany, take the train in 2nd class. For train rides longer than 4 hours, you may book 1st class. Flights within Germany are only allowed if the train takes more than 6 hours.
3. Hotels: The company pays up to 110 euros per night. In Munich, Frankfurt, and Hamburg, the limit is 150 euros per night. Breakfast is included in the limit.
4. Meals: For each full travel day, you get a daily allowance of 28 euros. For the day of arrival and the day of departure, you get 14 euros each.
5. Taxis: Taxis are only paid between 10 pm and 6 am, or if you carry equipment that weighs more than 15 kg.
6. Expense report: Submit all receipts in the expense tool SpendLog within 30 days after the trip. Receipts submitted later are not reimbursed."""

First, we ask the model *without* the document:

In [ ]:
question = "I am staying two nights in a hotel in Munich. How much does the company pay for the hotel at most?"

response = client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": question}], temperature=0
)
print(response.choices[0].message.content)

The model cannot know the answer. In the best case, it says so. In the worst case, it invents a plausible number.

## Part 2: The Document Goes into the Prompt

**In short: paste the document into the prompt, and tell the model to answer only from it.**

We use what you learned in 4.04:

* a **system prompt** with the rules for every answer,
* a **prompt template** with `###` blocks, and the document between `<document>` tags.

In [ ]:
SYSTEM_PROMPT = """You answer questions about a document.
Use only the information in the document. If the document does not contain the answer, say that you don't know."""

USER_TEMPLATE = """### Document
<document>
{document}
</document>

### Question
{question}"""


def build_messages(document, question):
    """Fills the template and returns the list of messages for the chat API."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(document=document, question=question)},
    ]


messages = build_messages(POLICY, question)

### What the model sees

In 4.02, Ollama showed us the finished text. The gateway does not: it applies the chat template on the server.

So we build a **mockup**: a small function that writes our messages into the chat template of the Qwen models, the way you saw it in 4.02. The real template of Qwen3.8 may add a few details, but the structure is the same.

In [ ]:
def what_the_model_sees(messages):
    """A mockup: writes the messages into the Qwen chat template from 4.02.
    The gateway hides the real template, so this is our reconstruction."""
    text = ""
    for message in messages:
        text += f"<|im_start|>{message['role']}\n{message['content']}<|im_end|>\n"
    text += "<|im_start|>assistant\n"  # the open assistant header: here the model starts writing
    print(text)


what_the_model_sees(messages)

Everything is one long text again: the system prompt, the document, the question, and the open assistant header at the end. The model continues this text, as always.

Now we send the same messages to the gateway:

In [ ]:
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
print(response.choices[0].message.content)
print(f"\n[{response.usage.prompt_tokens} prompt tokens, {response.usage.completion_tokens} generated tokens]")

With the document in the prompt, the model can answer: 2 nights × 150 euros = 300 euros.

### Your turn

1. Ask three questions of your own about the policy. One of them should have **no** answer in the document, for example "Can I bring my dog on a business trip?".
2. Does the model say that it doesn't know? If not, change `SYSTEM_PROMPT` and try again.

In [ ]:
# TODO: ask your own questions
my_questions = [
    "My train ride from Paderborn to Munich takes 4.5 hours. May I book 1st class?",
    "I travel from Monday to Wednesday. How much meal allowance do I get?",
    "Can I bring my dog on a business trip?",
]

for my_question in my_questions:
    response = client.chat.completions.create(model=MODEL, messages=build_messages(POLICY, my_question), temperature=0)
    print(my_question)
    print("->", response.choices[0].message.content, end="\n\n")

## Part 3: The Same with LangChain

**In short: LangChain wraps the OpenAI-compatible API and gives you building blocks that you can chain.**

[LangChain](https://docs.langchain.com/) is a Python library for building applications with LLMs. We use it for the rest of this session and in the next ones (RAG, tools, agents).

| Building block | What it does | Example |
|---|---|---|
| **Chat model** | sends messages to a model and returns the answer | `ChatOpenAI` |
| **Prompt template** | fills placeholders and returns the messages | `ChatPromptTemplate` |
| **Output parser** | turns the answer into what your code needs | `StrOutputParser` |

`ChatOpenAI` is LangChain's wrapper around the OpenAI-compatible API. It takes the same three things as the `OpenAI` client in 4.03: `base_url`, `api_key`, and the model name.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL, base_url=AI_GATEWAY_BASE, api_key=API_KEY, temperature=0)

`ChatOpenAI` even takes our list of messages from Part 2 directly:

In [ ]:
answer = llm.invoke(messages)

print(answer.content)
print(answer.usage_metadata)

The answer is no longer a raw API response but an `AIMessage`: the text is in `content`, the token counts are in `usage_metadata`.

### The prompt template in LangChain

Our template from Part 2 works in LangChain with almost no changes. `ChatPromptTemplate` takes a list of `(role, text)` pairs with placeholders. (LangChain calls the user role `human`.)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", USER_TEMPLATE),
])

prompt_value = prompt.invoke({"document": POLICY, "question": question})
prompt_value.to_messages()

The template returns the same messages as `build_messages()`. Our mockup shows that the model sees exactly the same text as in Part 2:

In [ ]:
from langchain_core.messages import convert_to_openai_messages

what_the_model_sees(convert_to_openai_messages(prompt_value.to_messages()))

### Chaining the blocks

The `|` operator connects the blocks into a **chain**. The output of one block is the input of the next:

1. `prompt` fills the template and returns the messages.
2. `llm` sends them to the gateway and returns an `AIMessage`.
3. `StrOutputParser()` takes the text out of the `AIMessage`.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

chain.invoke({"document": POLICY, "question": question})

Same prompt, same model, same answer. LangChain only saves you the plumbing, and you can swap every block: a different model, a different template, a different parser.

### Your turn

Change the chain so that the model answers **in German** and names the **number of the rule** it used, for example "(Regel 3)".

Change only the prompt template, not the line with the chain. Test it with two questions.

In [ ]:
# TODO: a new system prompt; the user template stays the same
german_system_prompt = """You answer questions about a document.
Use only the information in the document. If the document does not contain the answer, say that you don't know.
Always answer in German. At the end of your answer, name the number of the rule you used, for example "(Regel 3)"."""

german_prompt = ChatPromptTemplate.from_messages([
    ("system", german_system_prompt),
    ("human", USER_TEMPLATE),
])
german_chain = german_prompt | llm | StrOutputParser()

print(german_chain.invoke({"document": POLICY, "question": question}))
print(german_chain.invoke({"document": POLICY, "question": "Does the company pay my taxi at 11 pm?"}))

## Part 4: When the Document Gets Long

**In short: putting the whole document into every prompt does not scale.**

Our policy has a few hundred tokens. Real documents are much longer. In the next notebooks, we work with two Wikipedia articles: *Game of Thrones* and *How I Met Your Mother*.

In [ ]:
# downloads the two articles as Markdown, if they are not here yet
BASE_URL = "https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_05/"
for path in ["markdown_documents/Game_of_Thrones.md", "markdown_documents/How_I_Met_Your_Mother.md"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + path, path)

articles = {}
for path in ["markdown_documents/Game_of_Thrones.md", "markdown_documents/How_I_Met_Your_Mother.md"]:
    with open(path, encoding="utf-8") as file:
        articles[path] = file.read()
    # rule of thumb: one token is about 4 characters of English text
    print(f"{path}: {len(articles[path]):,} characters, about {len(articles[path]) // 4:,} tokens")

Let's send the whole *How I Met Your Mother* article with one question. (This takes a while and uses a good part of your token budget, so run it only once.)

In [ ]:
himym = articles["markdown_documents/How_I_Met_Your_Mother.md"]

start = time.time()
response = client.chat.completions.create(
    model=MODEL, messages=build_messages(himym, "Who plays Ted Mosby?"), temperature=0
)
print(response.choices[0].message.content)
print(f"\n[{response.usage.prompt_tokens:,} prompt tokens, {time.time() - start:.0f} seconds]")

The answer is right, but look at the price: tens of thousands of prompt tokens for a one-line answer, and **every** question costs the same again.

Compare the real number of prompt tokens with our estimate from above. The estimate is too low: links, numbers, and Markdown need more tokens than normal English text. The "4 characters per token" rule is only a rough guide.

Three problems:

* **Cost and time:** you pay for the whole document with every question.
* **Limits:** a collection of documents quickly gets larger than the context window of any model.
* **Lost in the middle:** models use information in the middle of a long context worse than information at the beginning or the end ([Liu et al., 2024](https://doi.org/10.1162/tacl_a_00638)).

**The idea of RAG:** do not send the whole document. First **retrieve** only the few pieces that are relevant for the question, and then let the model **generate** the answer from them. That is the next notebook.

## Questions?

Anything unclear about documents in prompts, the template mockup, or LangChain? Ask now.

## What You Learned

* A model only knows its training data and what is in the prompt. For your own documents, the document has to go into the prompt.
* A **system prompt** plus a **template** with `###` blocks and `<document>` tags gives the model the document, the question, and the rules.
* What the model sees is still **one long text**: the gateway hides the template, so we built a mockup.
* **LangChain** wraps the API in building blocks: `ChatOpenAI` (model), `ChatPromptTemplate` (template), `StrOutputParser` (parser). The `|` operator chains them.
* Whole documents in every prompt are expensive, limited by the context window, and get "lost in the middle". **RAG** sends only the relevant pieces.

Next: **5.02 RAG Basics**. We build retrieval-augmented generation from scratch.